In [ ]:
import ray
from ray.train import ScalingConfig, RunConfig, Checkpoint
from ray.train.torch import TorchTrainer
import torch
import torch.nn as nn
import torch.nn.functional as F
import tempfile
import os
import numpy as np
import pandas as pd

# Re-train (Update) Recommender using Ray Train

We want to update and finetune our recommender model periodically (perhaps very frequently) using recent interaction data from our users.

Let's see how to train this model using Ray Data and Ray Train.

We'll start with the basic PyTorch code and local training, to see the original structure.

In [ ]:
class TwoTower(nn.Module):
    def __init__(self, num_users: int, num_items: int, dim: int = 64):
        super().__init__()
        self.user_emb = nn.Embedding(num_users, dim)
        self.item_emb = nn.Embedding(num_items, dim)

        # optional: small projection MLPs (kept minimal)
        self.user_proj = nn.Identity()
        self.item_proj = nn.Identity()

    def encode_users(self, user_ids: torch.LongTensor) -> torch.Tensor:
        u = self.user_proj(self.user_emb(user_ids))
        return F.normalize(u, dim=-1)

    def encode_items(self, item_ids: torch.LongTensor) -> torch.Tensor:
        v = self.item_proj(self.item_emb(item_ids))
        return F.normalize(v, dim=-1)

    def forward(self, user_ids: torch.LongTensor, pos_item_ids: torch.LongTensor):
        """
        Returns logits matrix [B,B] where diagonal is the positive pair and
        off-diagonals are in-batch negatives.
        """
        u = self.encode_users(user_ids)         # [B, D]
        v = self.encode_items(pos_item_ids)     # [B, D]
        logits = u @ v.t()                      # [B, B]
        return logits

In [ ]:
def train_loop():
    device = "cuda" if torch.cuda.is_available() else "cpu"
    num_users, num_items, dim = 1000, 1000, 64
    model = TwoTower(num_users, num_items, dim).to(device)
    opt = torch.optim.Adam(model.parameters(), lr=1e-3)

    # fake training data: (user, positive_item)
    for step in range(200):
        B = 256
        user_ids = torch.randint(0, num_users, (B,), device=device)
        pos_item_ids = torch.randint(0, num_items, (B,), device=device)

        opt.zero_grad()

        logits = model(user_ids, pos_item_ids)     # [B,B]
        targets = torch.arange(logits.size(0), device=logits.device)  # diagonal
        loss = F.cross_entropy(logits, targets)

        loss.backward()
        opt.step()
    
        if step % 50 == 0:
            print(f"step={step} loss={loss.item():.4f}")

In [ ]:
train_loop()

## Minimal port onto Ray Train

The following is a very minimal port onto Ray Train for distributed Torch DDP training.

There are a number of elements we'll want to improve, but the core conversion from Torch to Ray Train + Torch DDP is straightforward.

Note that `prepare_model` manages device detection and model/data movement to/from devices as well performs the `DDP(model)` wrapping that we would code for Torch distributed.

> this training loop will be deployed into a training process in each worker

In [ ]:
def train_loop_ray_minimal():
    num_users, num_items, dim = 1000, 1000, 64
    model = TwoTower(num_users, num_items, dim)  # .to(device) <------ device detection and management implicit
    
    model = ray.train.torch.prepare_model(model) # <------ wrap model, manage devices

    opt = torch.optim.Adam(model.parameters(), lr=1e-3)

    # fake training data: (user, positive_item)
    for step in range(200):
        B = 256
        user_ids = torch.randint(0, num_users, (B,)) # <------ device detection and management implicit
        pos_item_ids = torch.randint(0, num_items, (B,)) # <------ device detection and management implicit

        opt.zero_grad()

        logits = model(user_ids, pos_item_ids)     # [B,B]
        targets = torch.arange(logits.size(0), device=logits.device)
        loss = F.cross_entropy(logits, targets)

        loss.backward()
        opt.step()
        
        if step % 50 == 0:
            print(f"step={step} loss={loss.item():.4f}")

To orchestrate the training, we use `TorchTrainer` with a minimal config pointing to the training loop code, `ScalingConfig`, and a symmetric read/write storage path.

In [ ]:
trainer = TorchTrainer(train_loop_ray_minimal, 
                       scaling_config=ScalingConfig(num_workers=2), 
                       run_config=ray.train.RunConfig(storage_path='/mnt/cluster_storage'))

result = trainer.fit()

result

We'll re-write this training loop to add two improvements:
* parametrizing some values using a `config` dict that is supplied from our orchestration code, supporting better separation of concerns
* add model checkpointing and reporting of stats through Ray Train APIs
  * note that in some frameworks (e.g., Lightning) this checkpointing and reporting does not need to be coded by the user

In [ ]:
def train_loop_ray_checkpoint_stats_and_config(config):
    
    num_users, num_items, dim = config['num_users'], config['num_items'], config['dim']
    model = TwoTower(num_users, num_items, dim) # values from config
    
    model = ray.train.torch.prepare_model(model)

    opt = torch.optim.Adam(model.parameters(), lr=1e-3)

    # fake training data: (user, positive_item)
    for step in range(200):
        B = 256
        user_ids = torch.randint(0, num_users, (B,))
        pos_item_ids = torch.randint(0, num_items, (B,))

        opt.zero_grad()

        logits = model(user_ids, pos_item_ids)     # [B,B]
        targets = torch.arange(logits.size(0), device=logits.device)
        loss = F.cross_entropy(logits, targets)

        loss.backward()
        opt.step()
        
        if step % 50 == 0:
            with tempfile.TemporaryDirectory() as temp_checkpoint_dir:
                checkpoint = None

                # In standard DDP training, where the model is the same across all ranks,
                # only the global rank 0 worker needs to save and report the checkpoint
                if ray.train.get_context().get_world_rank() == 0:
                    torch.save(
                        model.module.state_dict(),  # NOTE: Unwrap the model.
                        os.path.join(temp_checkpoint_dir, "model.pt"),
                    )
                    checkpoint = Checkpoint.from_directory(temp_checkpoint_dir)

                ray.train.report({'loss': loss.item()}, checkpoint=checkpoint)    

The `config` values are supplied from the driver/orchestrator in the `TorchTrainer` constructor

In [ ]:
trainer = TorchTrainer(train_loop_ray_checkpoint_stats_and_config, 
                       scaling_config=ScalingConfig(num_workers=2), 
                       run_config=ray.train.RunConfig(storage_path='/mnt/cluster_storage'),
                       train_loop_config={'num_users' : 1000, 
                                          'num_items' : 1000, 
                                          'dim' : 64 })

result = trainer.fit()

result

## Integrate Ray Data pipeline for supplying training data

We would like to leverage the Ray Data + Ray Train integration in order to
* simplify, parallelize, and accelerate feature pre-processing for training
* take advantage of Ray's resource scheduling to operate on different hardware for data processing vs. training
* optimize and accelerate the delivery of training data batches into the training workers

We'll do this in two steps:

1. Create a Ray Data pipeline (Dataset) that represents the featurized data we want to train on
2. Adjust our training worker code to consume batches of data from that pipeline

In [ ]:
ds = ray.data.read_json('/mnt/cluster_storage/ecom/users.ndjson', lines=True, file_extensions=['.ndjson'])

ds.take_batch(3)

We'll implement a miniature version of our Database Facade and we'll add a `__call__` method to let us use it in a Dataset pipeline.

> Production Note: in order to improve separation of concerns, various patterns can split existing logic (such as database access code) away from the Ray Data specific code (such as the batch formats or `__call__`). These techniques include inheritance, factory patterns, or custom decorators.

In [ ]:
class DatabaseFacade():
    def __init__(self, users):
        self.users = pd.read_json(users, lines=True)
        
    def users_for_ids(self, ids):
        return self.users[self.users['id'].isin(ids)]

    def __call__(self, batch):
        batch['user_indices'] = self.users_for_ids(batch['id']).index.values
        return batch

In [ ]:
ds.select_columns(['id', 'last_20_positive_item_interactions']) \
    .map_batches(DatabaseFacade, fn_constructor_args=['/mnt/cluster_storage/ecom/users.ndjson']) \
    .take_batch(3)

Our model expects a batch of inputs where each input is a user-product pair, so we want to explode each user and the user's 20 interactions into 20 pairs.

In [ ]:
def explode_interactions(batch):
    batch_size = len(batch['id'])
    interactions = np.concatenate(batch['last_20_positive_item_interactions'])
    user_indices = np.concatenate([np.repeat(index, 20) for index in batch['user_indices']])
    return { 'user_indices' : user_indices, 'interactions' : interactions }

In [ ]:
training_data_preprocessing_pipeline = ds.select_columns(['id', 'last_20_positive_item_interactions']) \
    .map_batches(DatabaseFacade, fn_constructor_args=['/mnt/cluster_storage/ecom/users.ndjson']) \
    .map_batches(explode_interactions)

training_data_preprocessing_pipeline.take(40)

At this point, we have a data pipeline that produces the shape and flavor of data on which we want to train.

__Integrating the Dataset(s) with Ray Train code__

We'll implement two sets of changes:
1. consume batches of data within the train worker code
2. supply `Dataset`(s) (pipelines) to the Ray Train orchestrator, in the `Trainer` constructor

At a more detailed level, step 1 will...
* shard the dataset so that each worker gets a unique slice of the data for training
* create an iterator that produces batches of data for training, with options to
  * specify batch size
  * control data format and data type
  * prefetch for extra buffering
  * shuffle
* typically yield batches in the form of Python dicts, which we will index into to obtain NumPy or Torch tensors

In [ ]:
def train_loop_ray_data(config):
    from ray.train import get_dataset_shard

    model = TwoTower(config['num_users'], config['num_items'], config['dim'])  # get values from config   
    model = ray.train.torch.prepare_model(model)
    opt = torch.optim.Adam(model.parameters(), lr=1e-3)

    train_sh = get_dataset_shard("train") # <--- get shard of data stream for the present worker
    training = train_sh.iter_torch_batches(batch_size=1000) # <--- get iterator of torch batches

    for batch in training:
        user_ids = batch['user_indices']
        pos_item_ids = batch['interactions']
    
        opt.zero_grad()

        logits = model(user_ids, pos_item_ids)
        targets = torch.arange(logits.size(0), device=logits.device)
        loss = F.cross_entropy(logits, targets)

        loss.backward()
        opt.step()
        
        with tempfile.TemporaryDirectory() as temp_checkpoint_dir:
            checkpoint = None
            if ray.train.get_context().get_world_rank() == 0:
                torch.save(
                    model.module.state_dict(),
                    os.path.join(temp_checkpoint_dir, "model.pt"),
                )
                checkpoint = Checkpoint.from_directory(temp_checkpoint_dir)

            ray.train.report({'loss': loss.item()}, checkpoint=checkpoint)    

In [ ]:
trainer = TorchTrainer(train_loop_ray_data, 
                       scaling_config=ScalingConfig(num_workers=2), 
                       run_config=ray.train.RunConfig(storage_path='/mnt/cluster_storage'),
                       train_loop_config={'num_users' : 1000, 
                                          'num_items' : 1000, 
                                          'dim' : 64 },
                       datasets={'train' : training_data_preprocessing_pipeline}) # <--- supply training data pipeline

result = trainer.fit()

result

Now that we can feed batches to our training, we can refactor a bit to produce an epoch/batch pattern, with reporting and checkpointing at each epoch.

In [ ]:
def train_loop_ray_data_epochs(config):
    from ray.train import get_dataset_shard

    model = TwoTower(config['num_users'], config['num_items'], config['dim'])  # get values from config   
    model = ray.train.torch.prepare_model(model)
    opt = torch.optim.Adam(model.parameters(), lr=1e-3)

    train_sh = get_dataset_shard("train")
    
    global_batch_size = config['global_batch_size']
    per_worker_batch_size = global_batch_size // ray.train.get_context().get_world_size()
    training = train_sh.iter_torch_batches(batch_size=per_worker_batch_size) # <--- calculate local batch size if desired

    for epoch in range(config['num_epochs']):
        
        for batch in training:
            
            user_ids = batch['user_indices']
            pos_item_ids = batch['interactions']

            opt.zero_grad()

            logits = model(user_ids, pos_item_ids)
            targets = torch.arange(logits.size(0), device=logits.device)
            loss = F.cross_entropy(logits, targets)

            loss.backward()
            opt.step()

        with tempfile.TemporaryDirectory() as temp_checkpoint_dir:
            checkpoint = None
            if ray.train.get_context().get_world_rank() == 0:
                torch.save(
                    model.module.state_dict(),  # NOTE: Unwrap the model.
                    os.path.join(temp_checkpoint_dir, "model.pt"),
                )
                checkpoint = Checkpoint.from_directory(temp_checkpoint_dir)

            ray.train.report({'loss': loss.item()}, checkpoint=checkpoint)    

In [ ]:
trainer = TorchTrainer(train_loop_ray_data_epochs, 
                       scaling_config=ScalingConfig(num_workers=2), 
                       run_config=ray.train.RunConfig(storage_path='/mnt/cluster_storage'),
                       train_loop_config={'num_users' : 1000, 
                                          'num_items' : 1000, 
                                          'dim' : 64,
                                          'num_epochs' : 5,
                                          'global_batch_size' : 1024 },
                       datasets={'train' : training_data_preprocessing_pipeline})

result = trainer.fit()

result

Note that in the multi-epoch training example, the Dataset is performing streaming execution -- including retrieving the data from a source -- for every epoch.

In many cases, that is the desired behavior. E.g., 
* the data may be so large that streaming and reprocessing represents a small cost compared to the cost of eliminating that work (e.g., storage, less optimal movement, etc.)
* or it may be necessary to perform some random, unique, or time-dependent computation (e.g., random data augmentation) producing new data values for each training epoch

However, in other cases, it may be possible to compute the dataset once and cache it in a place which is more local to the training cluster. In the latter case, we can compute the dataset and store it across the Ray Object Store (distributed memory but, in the case of a large dataset, likely spilling to a distributed disk cache), by calling `materialize` on the dataset.

In [ ]:
cached_materialized_data = training_data_preprocessing_pipeline.materialize()

trainer = TorchTrainer(train_loop_ray_data_epochs, 
                       scaling_config=ScalingConfig(num_workers=2), 
                       run_config=ray.train.RunConfig(storage_path='/mnt/cluster_storage'),
                       train_loop_config={'num_users' : 1000, 
                                          'num_items' : 1000, 
                                          'dim' : 64,
                                          'num_epochs' : 5,
                                          'global_batch_size' : 1024 },
                       datasets={'train' : cached_materialized_data})

result = trainer.fit()

result

Note the difference in the Ray Data logging output when training from the materialized dataset.

Now we can wrap this training program in a Ray or Anyscale Job, and schedule it to regularly consume new data, train or finetune an existing model, and produce new model checkpoints in a known location.